Kolmogorov-Smirnoff Test

PHYS4730, Fall 2025

Lab 16

In [ ]:
import numpy as np
import scipy.optimize as SciOpt
import scipy.stats as SciStat
from numpy import random as rnd
from numpy import linalg as LA
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams.update({'font.size': 18})

In [ ]:
### Define two sets of data...         ###
### both Weibull but different shapes  ###

n,m    = 500, 500

Shape1 = 2.0
Shape2 = 3.0 

rvvec1 = rnd.weibull(Shape1,(n,1)).flatten()      # <== "flatten" column vector into row vector
rvvec2 = rnd.weibull(Shape2,(m,1)).flatten()      # <== "flatten" column vector into row vector

In [ ]:
### Plot the two sets ###

nbins =  100
aa    =  0.0
bb    =  4.0

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.hist(rvvec1,bins=nbins, range=[aa,bb])
plt.xlabel("Weibull RV 1")
plt.xlim([0.0,3.0])

plt.subplot(1,2,2)
plt.hist(rvvec2,bins=nbins, range=[aa,bb])
plt.xlabel("Weibull RV 2")
plt.xlim([0.0,3.0])

plt.show()

In [ ]:
### Find ECDFs for two datasets       ###
### Note binning choices will affect  ### 
### precise KS-statistic and p-value  ### 
### determination.                    ###

nbins =  100
aa    =  0.0
bb    =  4.0

dx    = (bb-aa)/nbins

ecdfX = np.arange(aa,bb,dx)

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
ecdfY1,b,p = plt.hist(rvvec1, bins=nbins, range=[aa,bb], density=True, cumulative=True)
plt.xlabel("Weibull ECDF 1")

plt.subplot(1,2,2)
ecdfY2,b,p = plt.hist(rvvec2, bins=nbins, range=[aa,bb], density=True, cumulative=True)
plt.xlabel("Weibull ECDF 2")

plt.show()

In [ ]:
### make a plot overlaying cumulative probabilities ###

fig = plt.figure(figsize=(15,7.5))

plt.plot(ecdfX,ecdfY1,'r',linewidth=3)
plt.plot(ecdfX,ecdfY2,'b',linewidth=3)
plt.fill_between(ecdfX, ecdfY1, ecdfY2, color='g')

plt.grid()
plt.xlabel('Data value')
plt.ylabel('Cumulative probability')

plt.show()

In [ ]:
### plot the difference between the two ECDFs ###
### the supremum is the Kolmogorov statistic  ###

fig = plt.figure(figsize=(15,7.5))
plt.plot(ecdfX,abs(ecdfY1-ecdfY2));
plt.grid()
plt.xlabel('Data value');
plt.ylabel('Cumulative probability difference');

loc   = np.argmax(abs(ecdfY1-ecdfY2))
kstat = ecdfY1[loc] - ecdfY2[loc]
plt.text(2.10,0.09,"Kolmogorov statistic: %7.5f" % kstat)

plt.show()

In [ ]:
### Remake the overlay plot with illustration of statistic ###

fig = plt.figure(figsize=(15,7.5))

plt.plot(ecdfX,ecdfY1,'r',linewidth=3)
plt.plot(ecdfX,ecdfY2,'b',linewidth=3)
plt.fill_between(ecdfX, ecdfY1, ecdfY2, color='g')

plt.grid()
plt.xlabel('Data value')
plt.ylabel('Cumulative probability')

titletext = "Kolmogorov Statistic = %7.5f" % kstat
plt.title(titletext)
plt.plot([ecdfX[loc],ecdfX[loc]],[ecdfY1[loc], ecdfY2[loc]],'k',linewidth=10);

plt.show()

Now, how do we use the supremum $D$ to determine the chance probability that the distributions are the same? 

a) One-sided (PDF vs EPDF) comparison with data size $n$: Kolmogorov (1933) showed that in the limit of large $n$, the probability $P\left(\sqrt{n} D_n \leq \lambda \right)$ will converge to a CDF given by the <i>Kolmogorov distribution</i>
$$
P(\sqrt{n} D_n \leq \lambda) \longrightarrow 1 - 2 \sum_{k=1}^{\infty} (-1)^{(k-1)} e^{-2k^2\lambda^2}
$$
thus the $p$-value will be given by 
$$
%\begin{align}
%p = P(D_n \geq d_{obs}) && = && 1 - P(D_n \leq d_{obs}) \\
%                        && = && 2 \sum_{k=1}^{\infty} (-1)^{(k-1)} e^{-2k^2n d_{obs}^2}
%\end{align}
p = P(D_n \geq d_{obs})  =  1 - P(D_n \leq d_{obs}) \\
                         =  2 \sum_{k=1}^{\infty} (-1)^{(k-1)} e^{-2k^2n d_{obs}^2}
$$

b) Two-sided (EPDF vs EPDF) comparison with data sizes $n$, $m$: Define the effective sample size by

$$
N = \frac{nm}{n+m}
$$
then
$$
%\begin{align}
%p = P(D_N \geq d_{obs}) && = && 2 \sum_{k=1}^{\infty} (-1)^{(k-1)} e^{-2k^2N d_{obs}^2}
%\end{align}
p = P(D_N \geq d_{obs}) =  2 \sum_{k=1}^{\infty} (-1)^{(k-1)} e^{-2k^2N d_{obs}^2}
$$


In [ ]:
### try to compute the p-value ###

### (a) Find pval as f(d_obs) ###

xx = np.arange(0.0,3.0,0.0001)

N = (n*m)/(n+m)

pval = np.zeros(len(xx))
for k in np.arange(1,100,1):
    pval += 2.0*(-1.0)**(k-1)*np.exp(-2.0*k*k*N*xx*xx)  # this sum is tricky   
    
pval[0] = 1.0  # annoying to have to do this kludge
 

In [ ]:
### function to find nearest value ###

def find_nearest(array, value):
    
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx, array[idx]

In [ ]:
### try to compute the p-value ###

### (b) Find pval corresponding to our kstat ###

kindex, kvalue = find_nearest(xx,kstat)
mypval = pval[kindex]

print("p-value =",mypval)

In [ ]:
### make a plot ###
    
plt.figure(figsize=[9,9])
    
plt.plot(xx,pval,label='chance probability')

plt.axvline(x=kstat, color='r', linestyle='--', label='observed KS Statistic')

plt.xlabel("$d_{obs}$")
plt.ylabel("$Pr(D_N \geq d_{obs})$")
plt.yscale("log")
plt.xlim([0.0,0.2])
plt.ylim([0.000001,1.5])
plt.grid()
plt.legend(loc = "upper right")
plt.show()   

In [ ]:
### So, that's what the Kolmogorov statistic is.  ###
### Now, scipy.stats will do all of this for us,  ###
### right up to computing p-value.                ###
###                                               ### 
### There are a couple of ways of doing this.     ###
### Compare scipy results to our answer.          ###

output1 = SciStat.kstest(rvvec1,rvvec2,alternative='two_sided')
output2 = SciStat.ks_2samp(rvvec1,rvvec2)

print("")
print("Scipy Method 1:")
print("The Kolmogorov statistic is:     %7.5f" % output1.statistic)
print("The p-value is:                  %7.5e" % output1.pvalue)
print("")
print("Scipy Method 2:")
print("The Kolmogorov statistic is:     %7.5f" % output2.statistic)
print("The p-value is:                  %7.5e" % output2.pvalue)
print("")
print("Our method:")
print("The Kolmogorov statistic is:     %7.5f" % kstat)
print("The p-value is:                  %7.5e" % mypval)
print("")
